In [2]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


In [3]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
 C    0.879477   -0.347438    0.000000
 C    0.000000    0.675064    0.000000
 C   -0.879440   -0.347453    0.000000
 H    1.829361   -0.828858    0.000000
 H   -0.000385    1.776911    0.000000
 H   -1.829195   -0.829095    0.000000'''
molecule = 'C3H3_an'
spin = 0
charge =  -1


In [4]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 4 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [5]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [6]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./C3H3_an.txt
orbital  energy   occ
0     -10.872685688416338    2.0
1     -10.671433481526192    2.0
2     -10.671254053116089    2.0
3     -0.6961170365556837    2.0
4     -0.41784977703901593    2.0
5     -0.3984830991701245    2.0
6     -0.2967733997970341    2.0
7     -0.1588395603648185    2.0
8     -0.07809325932559512    2.0
9     0.08140341225421333    2.0
10     0.239009442755851    2.0
11     0.6640816233307193    0.0
12     0.7409501651928673    0.0
13     0.9196833120603742    0.0
14     0.9642606954942392    0.0
15     0.9790350212619463    0.0
16     1.3499989418295832    0.0
17     1.568972425160202    0.0
Homo_index: 10, Lumo_index: 11


WARN: System has nonzero charge -1; the dipole moment is origin-dependent.
Location of origin: [0. 0. 0.]


In [8]:
n_elec= 4
n_orb = 3 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([9,11,12])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-114.56428131322185),
 np.float64(-2.086553505783783),
 FCIvector([[ 9.60401577e-01,  2.29494937e-05, -6.79764761e-02],
            [ 2.29494937e-05, -2.38926988e-01,  7.42256772e-06],
            [-6.79764761e-02,  7.42256772e-06, -1.06306638e-01]]),
 NPArrayWithTag([[ 3.42655830e-03, -7.18501631e-01,  6.88697421e-01,
                  -1.23762543e-01, -1.38130977e-01,  8.42226019e-02,
                  -1.11518352e-02, -3.94491126e-02,  2.24314523e-02,
                   0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
                  -1.18141595e-01,  1.22055320e-01,  4.11971324e-02,
                   1.07128561e-01,  3.35356937e-02, -3.23488097e-02],
                 [-4.61015065e-03, -1.70528741e-02,  1.40332746e-02,
                   3.06672255e-01,  4.16224281e-01, -2.53947403e-01,
                   5.99427059e-02,  1.46620223e-01, -1.48033735e-01,
                   0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
                   6.39611220e-01, -8.1366350

In [9]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1]        [0 1]        0.960401577104013
[0 1]        [0 2]        0.000022949493685
[0 1]        [1 2]        -0.067976476055451
[0 2]        [0 1]        0.000022949493686
[0 2]        [0 2]        -0.238926988096034
[0 2]        [1 2]        0.000007422567718
[1 2]        [0 1]        -0.067976476055451
[1 2]        [0 2]        0.000007422567718
[1 2]        [1 2]        -0.106306638071668
